# Sổ thực hành DataFusion — tự viết mã

Notebook này chỉ có hướng dẫn và ô mã trống. Không có lời giải, kết quả chạy sẵn hoặc chương trình thực nghiệm hoàn chỉnh.

Mục tiêu: tự viết từng bước, quan sát DataFusion thật và giải thích được kết quả. Môi trường dự kiến: Homebrew Python 3.13.15, macOS ARM64. Chọn bộ thực thi **DataFusion · Python 3.13.15** theo README trước khi bắt đầu.

Các thí nghiệm phục vụ học tập và nghiên cứu bài báo; không nhận các ý tưởng hoặc kết quả của tác giả là đóng góp mới.

## Bài báo có mã không? Phần nào là chạy thật?

- **Hình 3, mục 5.5.1:** đoạn Rust minh họa cách toán tử xử lý luồng. Đây không phải một chương trình Python hoàn chỉnh để chạy ngay.
- **Mục 8:** thí nghiệm đo hiệu năng thật. Chú thích 3 dẫn đến [kho mã thí nghiệm của tác giả](https://github.com/JayjeetAtGithub/datafusion-duckdb-benchmark). Kho ghi DataFusion 32.0.0 và DuckDB 0.9.1.
- **Notebook này:** khi tự viết lời gọi thư viện DataFusion, truy vấn sẽ được lập kế hoạch và thực thi bởi DataFusion thật. Dữ liệu tự sinh không biến bộ máy truy vấn thành giả lập.
- **Trang HTML của dự án:** các hoạt ảnh là mô phỏng giải thích cơ chế, không trực tiếp chạy DataFusion.

Nguồn: [bài báo gốc](../00-paper/Apache_Arrow_DataFusion.pdf), Hình 3 ở trang in 9 (trang PDF 5), mục 8 ở trang in 12–13 (trang PDF 8–9); [DOI](https://doi.org/10.1145/3626246.3653368).

D1–D8 là lộ trình học do người soạn xây dựng từ nội dung bài báo, không phải tám bài thực hành do tác giả cung cấp.

## Cách học với notebook

1. Đọc yêu cầu ngay trên ô mã, tự viết và chạy bằng **Shift + Enter**.
2. Chỉ chuyển tiếp khi đã kiểm tra điều kiện hoàn thành của bước hiện tại.
3. Nếu có lỗi, đọc thông báo và kiểm tra tên biến, kiểu dữ liệu, đường dẫn, bộ thực thi; không chạy lại mọi ô một cách ngẫu nhiên.
4. Sau mỗi minh họa, dùng ô ghi nhận để viết cách hiểu và bằng chứng của chính lần chạy.
5. Cuối cùng, khởi động lại bộ thực thi và chạy từ đầu để phát hiện sự phụ thuộc vào các ô chạy sai thứ tự.

Phân biệt: JupyterLab là giao diện; notebook là tệp `.ipynb`; bộ thực thi Python là tiến trình chạy mã. Jupyter không phải bộ máy truy vấn.

## I. Kiểm tra môi trường

### 1. Phiên bản và vị trí thư mục

**Tự viết:** nhập các thư viện chuẩn để kiểm tra phiên bản Python, kiến trúc máy, thư mục làm việc; nhập DataFusion, PyArrow, NumPy và DuckDB, rồi kiểm tra phiên bản thư viện.

**Gợi ý tra cứu:** `sys.version`, `sys.executable`, `platform.machine`, `Path.cwd`, `importlib.metadata.version`.

**Điều kiện hoàn thành:** Python 3.13.15, kiến trúc `arm64`, trình thông dịch nằm trong `.venv` của dự án; thư mục làm việc là `04-demo`. Nếu không đúng, chọn lại bộ thực thi trước khi cài hoặc chạy gì thêm.

**Cần giải thích:** vì sao Python ngoài Terminal có thể khác Python trong notebook?

## II. Tự tạo dữ liệu dùng chung

### 1. Bảng nhỏ để kiểm tra trước

**Tự viết:** chọn N = 20.000 để bắt đầu. Tạo bảng gồm ba cột: `a` là số nguyên từ 0 đến N−1; `c` bằng hai lần `a`; `b` là một trong A đến G, chia miền `a` thành bảy khoảng liên tiếp gần bằng nhau.

**Gợi ý:** dùng NumPy để tạo dãy số; chỉ số chữ cái là phần nguyên của `7 × a / N`; dùng PyArrow để tạo bảng với kiểu số nguyên 64 bit và chuỗi.

**Điều kiện hoàn thành:** đúng N dòng, `a` không trùng, `c = 2 × a`, `b` tăng từ A đến G theo `a`.

**Cần giải thích:** dữ liệu đã sắp theo `a` có đồng thời đúng thứ tự `(b, a)` không? Vì sao?

### 2. Hai tệp có cùng dữ liệu, khác thứ tự

**Tự viết:** tạo thư mục dữ liệu của riêng lần học. Ghi một bản giữ nguyên thứ tự và một bản xáo trộn toàn bộ dòng với hạt giống cố định. Khi ghi Parquet, chọn 2.000 dòng mỗi nhóm để bảng nhỏ có 10 nhóm.

**Gợi ý tra cứu:** `Path.mkdir`, `numpy.random.default_rng`, `permutation`, `Table.take`, `pyarrow.parquet.write_table`, tham số `row_group_size`.

**Điều kiện hoàn thành:** hai tệp cùng số dòng và cùng các bộ giá trị; không xáo trộn từng cột riêng vì như vậy sẽ phá quan hệ giữa `a`, `b`, `c`. Không ghi đè tệp của người khác.

Sau khi bảng nhỏ đúng, có thể tự đổi N thành 2.000.000 và nhóm dòng thành 200.000 để quan sát bộ nhớ và ghi tạm ra đĩa rõ hơn.

### 3. Đọc thống kê của nhóm dòng

**Tự viết:** đọc siêu dữ liệu Parquet và lập bảng gồm tên tệp, số thứ tự nhóm, số dòng, min/max của `a` và `b`.

**Gợi ý tra cứu:** `ParquetFile.metadata`, `num_row_groups`, `row_group`, `column`, `statistics.min`, `statistics.max`. Số dòng thuộc nhóm lấy từ `row_group.num_rows`, không từ đối tượng thống kê cột.

**Điều kiện hoàn thành:** giải thích được vì sao khoảng min/max thường hẹp ở bản đã sắp và rộng ở bản xáo trộn.

Giữ bảng thống kê này làm bằng chứng cho D4.

## III. Các minh họa

### D1. SQL và giao diện DataFrame

**Nguồn:** mục 5.3.3 của bài báo.

**Mục tiêu:** xem hai cách biểu diễn cùng yêu cầu có tạo cùng kế hoạch và cùng kết quả hay không.

**Bước 1 — tự viết:** tạo `SessionContext`, đăng ký tệp đã sắp dưới tên `t`; dùng SQL lấy `a`, `b` của các dòng có `a > N−3`.

**Gợi ý tra cứu:** `SessionContext`, `register_parquet`, `sql`, `collect`.

**Kiểm tra:** dữ liệu sinh đúng thì có hai giá trị `a` thỏa điều kiện. Đừng mặc định thứ tự kết quả khi chưa yêu cầu sắp xếp.

**Bước 2 — tự viết:** diễn đạt đúng yêu cầu trên bằng giao diện DataFrame, lấy kế hoạch logic đã tối ưu của cả hai cách và đối chiếu giá trị kết quả.

**Gợi ý tra cứu:** `table`, `filter`, `select`, `col`, `lit`, `optimized_logical_plan`, `display_indent`.

**Điều kiện hoàn thành:** cùng điều kiện lọc, cùng cột đầu ra; kiểm tra cả kế hoạch và dữ liệu, không chỉ nhìn số dòng.

**Cần giải thích:** vì sao SQL và DataFrame có thể dùng chung phần tối ưu và thực thi?

**Ghi nhận D1:** mô tả kết quả thực tế, dòng nào trong kế hoạch làm bằng chứng và điều gì chưa được thí nghiệm này chứng minh. Viết vào ô văn bản dưới đây.

### D2. Vòng đời truy vấn

**Nguồn:** mục 5.1 và 6.1–6.3.

**Tự viết:** truy vấn lọc `a > N/2`, gom theo `b`, tính tổng `c`, sắp tổng giảm dần và lấy ba nhóm đầu. Đọc kế hoạch logic ban đầu, logic sau tối ưu và kế hoạch vật lý.

**Gợi ý tra cứu:** `EXPLAIN VERBOSE`, cột `plan_type` và `plan`; lấy kết quả bằng `collect`. Với DataFusion 50.1.0, không dùng `show` để xử lý `EXPLAIN` vì có thể làm câu lệnh giải thích không còn ở gốc kế hoạch.

**Kiểm tra:** tìm phép lọc, phép gom nhóm, phép sắp xếp và giới hạn ba kết quả. Tìm `TopK`, `Partial`, `FinalPartitioned`, `RepartitionExec` nếu cấu hình thực tế tạo các toán tử đó.

**Cần giải thích:** kế hoạch logic nói làm gì, kế hoạch vật lý nói làm thế nào; không phải mọi truy vấn gom nhóm đều bắt buộc dùng hai pha.

**Ghi nhận D2:** chọn ba dòng trong kế hoạch và giải thích ý nghĩa; phân biệt quan sát thực tế với kiến thức từ bài báo.

### D3. Lô dữ liệu và phân vùng

**Nguồn:** mục 5.5.1–5.5.2.

**Bước 1 — tự viết:** đọc một cột, đếm số dòng của từng `RecordBatch`, lập bảng kích thước lô và số lượng lô.

**Gợi ý tra cứu:** `RecordBatch.num_rows`, `SessionConfig.with_batch_size`.

**Kiểm tra:** tổng số dòng trong các lô bằng N. Mặc định hướng đến 8.192 dòng một lô; không yêu cầu mọi lô đều đủ 8.192 dòng.

**Bước 2 — tự viết:** tạo hai phiên truy vấn độc lập, lần lượt đặt một và bốn phân vùng đích. So kế hoạch của cùng một truy vấn gom nhóm.

**Gợi ý tra cứu:** `SessionConfig.with_target_partitions`, `AggregateExec`, `RepartitionExec`.

**Cần giải thích:** phân vùng là cách chia công việc; không đồng nghĩa một phân vùng luôn chiếm đúng một lõi. Có thêm chi phí phối hợp nên không hứa số phân vùng tăng bao nhiêu thì tốc độ tăng bấy nhiêu.

**Ghi nhận D3:** ghi tổng dòng, các kích thước lô, điểm khác biệt giữa hai kế hoạch.

### D4. Bỏ nhóm dòng và lọc trong bộ đọc Parquet

**Nguồn:** mục 6.8.

**Bước 1 — tự viết:** chọn các dòng có `a > 3N/4` và `b` bằng F; gom theo `b`, đếm dòng và lấy giá trị lớn nhất của `c`. Chạy cùng yêu cầu trên hai tệp.

**Kiểm tra:** giá trị phải trùng nhau vì hai tệp chỉ khác thứ tự dòng. Đối chiếu khoảng `a` phù hợp với bảng dữ liệu tự sinh trước khi tin kết quả.

**Bước 2 — tự viết:** chạy bốn cấu hình: đã sắp/xáo trộn kết hợp với tắt/bật bộ lọc tại bộ đọc Parquet. Giữ các điều kiện khác giống nhau. Dùng `EXPLAIN ANALYZE` lấy chỉ số; nếu đo thời gian riêng, ghi rõ đó là một lần chạy khác.

**Gợi ý tra cứu:** cấu hình `datafusion.execution.parquet.pushdown_filters`; ở `DataSourceExec`, đọc `row_groups_pruned_statistics`, `row_groups_matched_statistics`, `pushdown_rows_pruned`, `bytes_scanned`, `output_rows`. Nếu thiếu chỉ số, ghi chưa có thông tin thay vì tự gán bằng 0.

**Cần giải thích:** bỏ cả nhóm dòng dựa vào min/max khác với lọc từng dòng sau giải mã. Bản xáo trộn có thể không bỏ được nhóm nào nhưng vẫn lọc bỏ nhiều dòng. `bytes_scanned` không tự chứng minh số byte đọc từ đĩa vật lý.

**Giới hạn:** các chỉ số này minh họa hoạt động của bộ đọc; không quan sát trực tiếp mọi bước giải mã nội bộ mà mục 6.8 mô tả.

**Ghi nhận D4:** lập bảng bốn cấu hình, đối chiếu min/max và giải thích từng chỉ số; chưa kết luận bật bộ lọc luôn nhanh hơn.

### D5. Tận dụng thứ tự đã có

**Nguồn:** mục 6.1 và 6.7.

**Tự viết:** chỉ khai báo thứ tự `(b, a)` cho tệp thật sự đã sắp. So kế hoạch sắp xếp và gom theo `(b, a)` với tệp xáo trộn.

**Gợi ý tra cứu:** `CREATE EXTERNAL TABLE`, `WITH ORDER`, `SortExec`, `ordering_mode=Sorted`, `peak_mem_used`.

**Kiểm tra:** xem có loại bỏ sắp xếp thừa không; so bộ nhớ của đúng toán tử. Không coi `peak_mem_used` là tổng RAM của Python.

**Cảnh báo:** khai báo thứ tự sai có thể dẫn đến kết quả sai. Phải kiểm tra dữ liệu, không khai báo chỉ để có một kế hoạch đẹp.

**Ghi nhận D5:** ghi bằng chứng về thứ tự và bộ nhớ; nêu rõ đã đối chiếu bao nhiêu dòng/nhóm kết quả.

### D6. Thiếu bộ nhớ và ghi tạm ra đĩa

**Nguồn:** mục 5.5.4, 6.2 và 7.4.

**Tự viết:** sắp xếp toàn bộ bảng xáo trộn theo `c` giảm dần và `b`. Giữ cùng hạn mức bộ nhớ quản lý, so hai môi trường cho phép/không cho phép ghi tạm ra đĩa.

**Gợi ý tra cứu:** `RuntimeEnvBuilder`, `with_fair_spill_pool`, `with_disk_manager_os`, `with_disk_manager_disabled`; chỉ số `spill_count`, `spilled_bytes` ở `SortExec`.

**Kiểm tra:** bắt riêng lỗi thiếu bộ nhớ dự kiến khi tắt đĩa; lỗi cú pháp hoặc đường dẫn không phải bằng chứng của cơ chế này. Bảng 20.000 dòng có thể vừa bộ nhớ; thử bảng lớn hơn sau khi dữ liệu đã đúng.

**Cần giải thích:** hạn mức 20 MiB của vùng nhớ quản lý không phải giới hạn tổng RAM của tiến trình. Không cần và không nên làm máy cạn toàn bộ RAM để minh họa.

**Ghi nhận D6:** điều kiện phát sinh ghi tạm, chỉ số quan sát được, lỗi dự kiến và trường hợp chưa cần ghi tạm.

### D7. Hàm do người dùng định nghĩa

**Nguồn:** mục 7.1.

**Bước 1 — tự viết:** hàm nhận mảng Arrow số nguyên, nhân đôi từng giá trị rồi trả lại mảng Arrow. Đăng ký vào DataFusion.

**Gợi ý tra cứu:** `udf`, kiểu đầu vào/đầu ra `pa.int64`, `pyarrow.compute.multiply`, tính chất `immutable`, `register_udf`. Hàm tính toán phải trả cùng kết quả với cùng đầu vào; nếu thêm bộ đếm, đó chỉ là dụng cụ quan sát.

**Bước 2 — tự viết:** so tổng sau khi gọi hàm với tổng của biểu thức nhân đôi có sẵn. Đếm số lần gọi hàm và tổng số dòng đưa vào hàm; dùng một phân vùng để phép đếm dễ theo dõi.

**Kiểm tra:** cả hai tổng bằng `N × (N−1)`; hàm nhận đủ N dòng nhưng được gọi theo lô thay vì một lần mỗi dòng.

**Cần giải thích:** hàm Python tham gia bộ máy thật nhưng có chi phí gọi sang Python. Thí nghiệm này không chứng minh hàm Python có hiệu năng bằng hàm Rust.

**Ghi nhận D7:** kết quả, số lần gọi, số dòng được xử lý và giới hạn của phép so sánh.

### D8. So sánh nhỏ với DuckDB

**Nguồn:** mục 8; tham khảo kho mã thí nghiệm của tác giả, không sao chép toàn bộ chương trình.

**Bước 1 — tự viết:** dùng phần mở rộng TPC-H của DuckDB sinh dữ liệu ở SF 0,01 trước, xuất tám bảng ra Parquet. Khi đã đúng có thể thử SF 0,1.

**Gợi ý tra cứu:** `INSTALL tpch`, `LOAD tpch`, `dbgen`, `tpch_queries`, xuất Parquet. Các bảng là `customer`, `orders`, `lineitem`, `part`, `partsupp`, `region`, `supplier`, `nation`.

**Kiểm tra:** không cần kết nối cơ sở dữ liệu trực tuyến hoặc nhập mật khẩu. Lần đầu tải phần mở rộng cần mạng; không đưa dữ liệu lớn hoặc thông tin đăng nhập lên Git.

**Bước 2 — tự viết:** đăng ký đúng cùng các tệp cho hai bộ máy. Chọn một truy vấn ngắn như Q6 trước. Đối chiếu số cột, số dòng, toàn bộ giá trị và số lần lặp của các dòng.

**Kiểm tra:** cùng số dòng không có nghĩa là cùng kết quả. Với số thực, nêu sai số chấp nhận; với truy vấn không quy định thứ tự, không báo khác chỉ vì dòng hiển thị ở vị trí khác.

Chỉ đo hiệu năng khi phép đối chiếu đã đúng.

**Bước 3 — tự viết:** đặt một phân vùng đích cho DataFusion, một luồng theo cấu hình DuckDB; mỗi truy vấn chạy năm lượt, bỏ hai lượt đầu và lấy trung vị ba lượt còn lại. Đổi lượt chạy trước giữa hai bộ máy để giảm thiên lệch thứ tự. Quy định rõ khoảng đo từ gửi SQL đến nhận đủ kết quả, sử dụng cùng kiểu kết quả đầu ra nếu có thể.

**Gợi ý tra cứu:** `time.perf_counter`, `statistics.median`; ghi cấu hình và số liệu vào bảng tự tạo.

**Cần giải thích:** đây không phải ràng buộc cứng tiến trình vào một lõi. Không chủ động xóa bộ nhớ đệm thì không gọi phép đo là đọc đĩa hoàn toàn chưa được lưu đệm. Không gán sẵn bộ máy thắng.

**Giới hạn quan trọng:** bài báo dùng TPC-H SF 10; máy, phiên bản và quy mô ở đây khác. Kết quả nhỏ không tái lập kết luận của bài báo.

**Ghi nhận D8:** phiên bản, cấu hình, quy mô, cách kiểm tra đúng, khoảng đo, thời gian, truy vấn lỗi và giới hạn. Chưa đo thì để trống, không chép thời gian của bài báo vào kết quả tự làm.

## IV. Tự kiểm tra trước khi trình bày

- Giải thích được bảng dữ liệu và điều kiện lọc mà không nhìn lời giải.
- Chỉ ra được bằng chứng trong kế hoạch thay vì chỉ đọc tên kỹ thuật.
- Phân biệt bỏ nhóm dòng, lọc dòng và giảm số cột đọc.
- Phân biệt kế hoạch logic, kế hoạch vật lý, lô dữ liệu và phân vùng.
- Phân biệt số liệu bài báo, số liệu tự đo và hoạt ảnh mô phỏng.
- Khởi động lại bộ thực thi rồi chạy từ đầu mà không phụ thuộc ô đã chạy ngầm.
- Không công khai đường dẫn cá nhân, thông tin đăng nhập hoặc dữ liệu không được phép chia sẻ.

## V. Nguồn để tra cứu

1. [Bài báo gốc](https://doi.org/10.1145/3626246.3653368) — các mục đã dẫn trong từng minh họa.
2. [Kho mã thí nghiệm của tác giả](https://github.com/JayjeetAtGithub/datafusion-duckdb-benchmark) — mã đo, truy vấn và kết quả gốc.
3. [DataFusion Python](https://datafusion.apache.org/python/) — cách dùng API; tài liệu hiện hành có thể khác bản 50.1.0.
4. [Phần mở rộng TPC-H của DuckDB](https://duckdb.org/docs/stable/extensions/tpch).
5. [JupyterLab](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html) — làm việc với ô mã và ô văn bản.

Khi viết báo cáo, dùng số trang in của bài báo và ghi nguồn theo quy định môn học; các mục D1–D8 không phải số mục nguyên bản của tác giả.